# ③ packed — NQ-Swap 외부 검증 (현경)

p8s에서 동결한 packed 사양(E17-6)을 NQ-Swap에 **그대로** 적용한다. 이 데이터에서 튜닝하는 값은 없다.

| 구성 | 값 (p8s에서 고정) |
|---|---|
| 입력 | [문맥 프롬프트] + [CB 프롬프트(BOS 제외)] 한 시퀀스, B는 BOS만 공유·위치 1부터 |
| steering | A 분기만, L14–18, Δ = ρ·MED·w, **ρ = 0.04** |
| 방향 w·MED | 이전 묶음의 −(h_A − h_B) 누적 평균 (**라벨 0개, 온라인**). 첫 묶음 steering 없음 |
| 선택 | X1: 첫 토큰이 갈리고 log P_A(top1_B) > log P_B(top1_A) 이면 B(closed-book) 분기 |
| 스트림 | 질문 순서 seed 0/1/2, 묶음 = 질문 8개(16행) |

## 다영 결과와 같은 조건

- 데이터 구성(`load_data`)·fold·프롬프트(`PP.build_messages`)·채점(`trunc` + `PP.norm`)·bootstrap(질문 단위, RandomState(0), 2,000회)을
  `ext_rho_sweep` / `ext_router_nqswap` 과 **글자 그대로** 맞췄다
- base·closed-book(K 라벨)·② always(ρ0.06)·① router 는 다영 체크포인트를 **불러온다** → 행 단위 paired 비교
- 다영 base 의 raw 를 이 노트북 채점기로 다시 매겨 em 이 전부 일치하는지 먼저 확인한다(관문)

## 실행 순서

맨 위 pip 셀 → [런타임 > 세션 다시 시작] → S1부터 끝까지.


## 0. 버전 고정 — 실행 후 세션 다시 시작

In [ ]:
!pip -q install "transformers==5.17.0" "accelerate==1.14.0"

## S1. 경로 · 설정 · 다영 체크포인트 찾기

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # CUDA 초기화 전
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import sys, json, time, hashlib, glob
from google.colab import drive
drive.mount("/content/drive")

MD     = "/content/drive/MyDrive/"
SRC = "/content/drive/.shortcut-targets-by-id/1fpN7NMJit_GxrQ1SWtfUeTPbRxkxpUQN/12주차_실험_다영/result/20261004T085540433485Z/source_data/"
P9  = MD + "Grad_share/data_share/P9/"
FRIEND = MD + "SOTA_다영/"                   # 다영 결과(base·closed-book·②·①)를 여기서 찾는다
OUT    = MD + "SOTA_다영/hk/results_ext/"
os.makedirs(OUT, exist_ok=True)

DATASET = "nqswap"
LIMIT   = 1000           # ← 다영 체크포인트와 같은 값이어야 paired 비교가 된다 (아래 '후보' 출력 확인)
TAG     = f"{DATASET}_n{LIMIT}"
SEED, N_FOLDS = 0, 5
LATE  = [14, 15, 16, 17, 18]
RHO   = 0.04             # p8s 동결값. 바꾸지 않는다
Q_PER_BATCH = 8          # OOM 이면 4 (보고서에 명시)
SEEDS = (0, 1, 2)

def _find(fname):
    hits = [h for h in sorted(glob.glob(FRIEND + f"**/{fname}", recursive=True)) if "/hk/" not in h]
    return hits
r_hits = _find(f"{TAG}_router.parquet")
PREF = os.path.dirname(r_hits[0]) + "/" if r_hits else None      # router 가 있는 폴더를 우선
FR = {}
for key, fn in [("base", "base"), ("closed_book", "closed_book"),
                ("always", "retain_transfer_r0.06"), ("router", "router")]:
    f = f"{TAG}_{fn}.parquet"
    FR[key] = PREF + f if PREF and os.path.exists(PREF + f) else (_find(f) or [None])[0]
    print(("OK   " if FR[key] else "없음 ") + f"{key:12s} {FR[key]}")

cand = sorted({os.path.basename(f).split("_")[1] for f in
               glob.glob(FRIEND + f"**/{DATASET}_n*_base.parquet", recursive=True)})
print(f"\n다영 {DATASET} 체크포인트 LIMIT 후보: {cand}  | 지금 TAG {TAG}")
for f in [P9 + "steer_v3/code/p8s_prompt.py", P9 + "steer_v3/env/manifest_cell1.json"]:
    print(("OK   " if os.path.exists(f) else "없음 ") + os.path.basename(f))
if not FR["base"]:
    print("\n[주의] 다영 base 가 없다 → 이 노트북이 base 를 직접 만든다(같은 방식). ②·① 비교는 빠진다")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
없음 base         None
없음 closed_book  None
없음 always       None
없음 router       None

다영 nqswap 체크포인트 LIMIT 후보: ['n1000', 'n300']  | 지금 TAG nqswap_n1000
OK   p8s_prompt.py
OK   manifest_cell1.json

[주의] 다영 base 가 없다 → 이 노트북이 base 를 직접 만든다(같은 방식). ②·① 비교는 빠진다


In [ ]:
## NQ-Swap 결과 파일 찾기 (CPU, 드라이브 크기에 따라 1~3분)
import glob, os, time, pandas as pd
MD = "/content/drive/MyDrive/"
hits = [h for pat in ["**/*nqswap*", "**/*nq_swap*", "**/*NQ-Swap*", "**/*NQSwap*"]
        for h in glob.glob(MD + pat, recursive=True)]
rows = []
for h in sorted(set(hits)):
    r = dict(path=h.replace(MD, ""), kb=round(os.path.getsize(h) / 1e3, 1) if os.path.isfile(h) else "dir",
             modified=time.strftime("%m-%d %H:%M", time.localtime(os.path.getmtime(h))))
    if h.endswith(".parquet"):
        try: r["rows"] = len(pd.read_parquet(h))
        except Exception as e: r["rows"] = f"읽기 실패 {type(e).__name__}"
    rows.append(r)
pd.set_option("display.max_colwidth", 200)
print(pd.DataFrame(rows).to_string(index=False))

# 노트북 안에 nqswap router/always 를 돌린 흔적이 있는지
print("\n[nqswap 을 다루는 노트북]")
for nb in glob.glob(MD + "**/*.ipynb", recursive=True):
    try:
        s = open(nb, encoding="utf-8").read()
        if "nqswap" in s and ("router" in s or "retain_transfer" in s):
            print(" ", nb.replace(MD, ""))
    except Exception:
        pass

                                                                                             path      kb    modified   rows
                                         12주차_실험_다영/result/20261004T085540433485Z/nqswap     dir 10-04 09:02    NaN
                            12주차_실험_다영/result/20261004T085540433485Z/nqswap_delta_ci.png    64.7 10-04 09:33    NaN
                        12주차_실험_다영/result/20261004T085540433485Z/nqswap_manifest.parquet  1423.6 10-04 08:56 2400.0
                             12주차_실험_다영/result/20261004T085540433485Z/nqswap_metrics.png    46.1 10-04 09:33    NaN
                              12주차_실험_다영/result/20261004T085540433485Z/nqswap_timing.csv     0.7 10-04 09:20    NaN
                          12주차_실험_다영/result/20261004T085540433485Z/nqswap_timing_raw.csv    35.5 10-04 09:20    NaN
                     12주차_실험_다영/result/20261004T085540433485Z/source_data/nqswap.parquet  1859.1 10-04 08:55 5510.0


## S2. 데이터 — 다영 `ext_rho_sweep` S2 와 동일

In [ ]:
## Cell F1 — 사라진 입력 파일 찾기 (CPU, Drive 크기에 따라 1~5분)
import os, unicodedata as ud
WANT = {"nqswap.parquet", "ConFiQA-QA.json", "p8s_prompt.py", "manifest_cell1.json"}
ROOTS = [r for r in ["/content/drive/MyDrive", "/content/drive/Shareddrives",
                     "/content/drive/.shortcut-targets-by-id"] if os.path.isdir(r)]
hits = {w: [] for w in WANT}
for R in ROOTS:
    for root, dirs, files in os.walk(R):
        for f in files:
            fn = ud.normalize("NFC", f)
            if fn in hits:
                hits[fn].append(os.path.join(root, f))
print("검색 루트:", ROOTS)
for w, hs in hits.items():
    print(f"\n[{w}] {len(hs)}개")
    for h in hs: print("  ", h)

검색 루트: ['/content/drive/MyDrive', '/content/drive/.shortcut-targets-by-id']

[ConFiQA-QA.json] 1개
   /content/drive/.shortcut-targets-by-id/1fpN7NMJit_GxrQ1SWtfUeTPbRxkxpUQN/12주차_실험_다영/result/20261004T085540433485Z/source_data/ConFiQA-QA.json

[nqswap.parquet] 1개
   /content/drive/.shortcut-targets-by-id/1fpN7NMJit_GxrQ1SWtfUeTPbRxkxpUQN/12주차_실험_다영/result/20261004T085540433485Z/source_data/nqswap.parquet

[p8s_prompt.py] 2개
   /content/drive/MyDrive/Grad_share/data_share/P9/steer_v3/code/p8s_prompt.py
   /content/drive/MyDrive/Grad_share/data_share/P9v4/inputs/p8s_prompt.py

[manifest_cell1.json] 1개
   /content/drive/MyDrive/Grad_share/data_share/P9/steer_v3/env/manifest_cell1.json


In [ ]:
import numpy as np, pandas as pd

def as_list(x):
    if isinstance(x, (list, tuple, np.ndarray, pd.Series)):
        return [str(v) for v in x if str(v).strip()]
    return [str(x)] if str(x).strip() else []

def load_data(name, limit=0):
    if name == "confiqa":
        df = pd.read_json(SRC + "ConFiQA-QA.json")
        n0 = len(df)
        df = df.drop_duplicates(subset=["question"], keep="first").copy()
        print(f"  질문 중복 제거 {n0} → {len(df)}")
        spec = dict(q="question", cpos="orig_context", cneg="cf_context",
                    fact="orig_answer", ctx="cf_answer", alias="orig_alias", qid=None)
    elif name == "nqswap":
        df = pd.read_parquet(SRC + "nqswap.parquet")
        print("  치환 유형:", {t: int((df.substitution_type == t).sum())
                              for t in df.substitution_type.unique()})
        df = df[df.substitution_type == "corpus"].copy()
        spec = dict(q="question", cpos="original_context", cneg="substituted_context",
                    fact="original_answers", ctx="substituted_answers", alias=None, qid="id")
    else:
        raise ValueError(name)

    qcol = spec["qid"] or spec["q"]
    vc = df[qcol].astype(str).value_counts()
    assert vc.max() == 1, f"질문당 {vc.max()}행이다: {list(vc.head(3).items())}"

    df = df.dropna(subset=[spec[k] for k in ("q", "cpos", "cneg", "fact")])
    print(f"  사용 가능 {len(df)}질문 (limit={limit or '전체'})")
    if limit:
        df = df.head(limit)

    rows = []
    for r in df.itertuples():
        q = str(getattr(r, spec["q"])).strip()
        gold = as_list(getattr(r, spec["fact"]))
        if spec["alias"]:
            gold += as_list(getattr(r, spec["alias"]))
        cans = as_list(getattr(r, spec["ctx"])) if spec["ctx"] else [""]
        if not gold:
            continue
        qid = str(getattr(r, spec["qid"])) if spec["qid"] else hashlib.md5(q.encode()).hexdigest()[:12]
        rows.append(dict(qid=qid, question=q, ctx_text=str(getattr(r, spec["cpos"])).strip(),
                         C="C+", gold=gold, ctx_answer=gold[0]))
        rows.append(dict(qid=qid, question=q, ctx_text=str(getattr(r, spec["cneg"])).strip(),
                         C="C-", gold=gold, ctx_answer=cans[0] if cans else ""))

    m = pd.DataFrame(rows).reset_index(drop=True)
    m["rid"] = np.arange(len(m))
    cnt = m.groupby("qid").size()
    assert (cnt == 2).all(), f"질문당 2행이 아닌 qid {int((cnt != 2).sum())}개"

    qids = sorted(m.qid.unique())
    rng = np.random.RandomState(SEED); rng.shuffle(qids)
    m["fold"] = m.qid.map({q: i % N_FOLDS for i, q in enumerate(qids)}).values
    return m

m = load_data(DATASET, LIMIT)
yC = (m.C == "C+").values
isCm = ~yC
fold = m.fold.values
N = len(m)
print(f"\n[{DATASET}] {m.qid.nunique()}질문 / {N}행")

  치환 유형: {'alias': 1186, 'corpus': 1685, 'popularity': 955, 'type_swap': 1684}
  사용 가능 1685질문 (limit=1000)

[nqswap] 1000질문 / 2000행


## S3. 모델 (A100 1~2분 / L4 2~3분)

In [ ]:
import torch, transformers
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
assert transformers.__version__ == "5.17.0", f"{transformers.__version__} — 세션 재시작"

sys.path.insert(0, P9 + "steer_v3/code")
import p8s_prompt as PP
REV = json.load(open(P9 + "steer_v3/env/manifest_cell1.json"))["model_revision"]

from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    from getpass import getpass
    login(token=getpass("HF read token: ").strip())

from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(PP.MODEL_ID, revision=REV)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(PP.MODEL_ID, revision=REV, dtype=torch.bfloat16,
            device_map="auto", attn_implementation="sdpa").eval()
EOS = sorted({tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")} - {None})
DEC = dict(skip_special_tokens=True, clean_up_tokenization_spaces=False)
print(PP.MODEL_ID, "| max_new", PP.MAX_NEW_TOKENS, "|", torch.cuda.get_device_name(0))

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

meta-llama/Llama-3.1-8B-Instruct | max_new 192 | NVIDIA A100-SXM4-40GB


## S4. 프롬프트 · 채점 · bootstrap — 다영 노트북과 동일

In [ ]:
import re
from collections import Counter
BS = 16

def build(q, ctx):
    return tok.apply_chat_template(PP.build_messages(q, ctx), tokenize=False, add_generation_prompt=True)

PROMPTS = np.array([build(r.question, r.ctx_text) for r in m.itertuples()], dtype=object)
NTOK = np.array([len(tok(p, add_special_tokens=False).input_ids) for p in PROMPTS])
FOLD_ROWS = {k: m[m.fold == k].assign(_t=NTOK[(m.fold == k).values])
                 .sort_values(["_t", "rid"], kind="stable").rid.values for k in range(N_FOLDS)}
print(f"{DATASET} | 프롬프트 토큰 중앙값 {int(np.median(NTOK))} 최대 {int(NTOK.max())}")

ABBR = r"(?<!\bDr)(?<!\bMr)(?<!\bMs)(?<!\bMrs)(?<!\bSt)(?<!\bJr)(?<!\bSr)(?<!\b[A-Z])"
def trunc(raw):
    s = str(raw).strip().split("\n")[0].strip()
    s = re.sub(r"^(answer|the answer is)\s*:?\s*", "", s, flags=re.I)
    if len(s.split()) > 8: s = re.split(ABBR + r"\.\s+(?=[A-Z])", s)[0]
    return re.sub(r"[.,;:\s]+$", "", s.strip().strip("\"'")).strip()

def rep3(t):
    w = str(t).split()
    g = Counter(tuple(w[i:i+3]) for i in range(len(w)-2))
    return int(bool(g) and max(g.values()) >= 3)

GOLD = list(m.gold)
CTXA = [PP.norm(x) for x in m.ctx_answer]

def score(raws, rids):
    rids = np.asarray(rids)
    p = [PP.norm(trunc(x)) for x in raws]
    d = pd.DataFrame(dict(rid=rids, raw=list(raws), pred=[trunc(x) for x in raws]))
    d["em"] = [int(any(pi == PP.norm(g) for g in GOLD[r])) for pi, r in zip(p, rids)]
    d["acc_x"] = [int(CTXA[r] != "" and pi == CTXA[r]) for pi, r in zip(p, rids)]
    d["rep3"] = d.raw.map(rep3)
    return d.sort_values("rid").reset_index(drop=True)

q = m.question.values
uq = np.unique(q); qidx = {u: np.where(q == u)[0] for u in uq}
rs = np.random.RandomState(0)
BOOT = [np.concatenate([qidx[u] for u in rs.choice(uq, len(uq))]) for _ in range(2000)]
def ci(v, mask=None):
    mask = np.ones(len(v), bool) if mask is None else mask
    return np.percentile([v[i][mask[i]].mean() for i in BOOT], [2.5, 97.5])
print("GOLD 예시", GOLD[0][:2], "| 문맥 주장 답(C-)", m.ctx_answer[1])

nqswap | 프롬프트 토큰 중앙값 230 최대 1176
GOLD 예시 ['Ray Charles'] | 문맥 주장 답(C-) Khrushchev


In [ ]:
## Cell G0 — 재현 관문: 저장된 base fold 0 재생성 → raw 문자열 대조 (A100 약 1분 / L4 2~3분)
@torch.no_grad()
def gen(prompts, max_new=None):                         # Cell G0: S5 정의와 동일
    out = []
    for s in range(0, len(prompts), BS):
        enc = tok(prompts[s:s+BS], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=max_new or PP.MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=EOS)[:, enc.input_ids.shape[1]:]
        for row in g.tolist():
            stop = next((i for i, t in enumerate(row) if t in EOS), None)
            out.append((tok.decode(row[:stop if stop is not None else len(row)], **DEC),
                        stop if stop is not None else len(row)))
    return out

SAVED = pd.read_parquet(OUT + f"{TAG}_base.parquet").set_index("rid")
assert len(SAVED) == N, f"저장 base {len(SAVED)}행 vs 현재 {N}행"

r0   = FOLD_ROWS[0]
new  = gen(PROMPTS[r0].tolist())                        # Cell G0: S5와 같은 배치 구성
same = np.array([raw == SAVED.raw[r] for (raw, _), r in zip(new, r0)])
print(f"fold 0 raw 일치 {same.sum()}/{len(r0)}")
for (raw, _), r in [x for x, s in zip(zip(new, r0), same) if not s][:5]:
    print(f"  rid {r}\n    저장: {SAVED.raw[r]!r}\n    지금: {raw!r}")
assert same.all(), "불일치 — S5 이후로 진행하지 말고 출력 붙여주기"
print("관문 통과 → S5부터 진행")

fold 0 raw 일치 400/400
관문 통과 → S5부터 진행


## S5. base · K 라벨 — 다영 체크포인트 불러오기 + 관문

관문: 다영 base 의 raw 를 이 노트북 채점기로 다시 매긴 em 이 **전부 일치**해야 한다.
일치하면 데이터 순서·채점이 같다는 뜻이므로 행 단위 paired 비교가 성립한다.

In [ ]:
@torch.no_grad()
def gen(prompts, max_new=None):
    out = []
    for s in range(0, len(prompts), BS):
        enc = tok(prompts[s:s+BS], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=max_new or PP.MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=EOS)[:, enc.input_ids.shape[1]:]
        for row in g.tolist():
            stop = next((i for i, t in enumerate(row) if t in EOS), None)
            out.append((tok.decode(row[:stop if stop is not None else len(row)], **DEC),
                        stop if stop is not None else len(row)))
    return out

# base
if FR["base"]:
    B0 = pd.read_parquet(FR["base"]).sort_values("rid").reset_index(drop=True)
    assert len(B0) == N, f"다영 base {len(B0)}행 vs {N}행 — LIMIT 확인"
    chk = score(B0.raw.tolist(), B0.rid.values)
    bad = int((chk.em.values != B0.em.values).sum())
    print(f"[관문] 다영 base 재채점 em 불일치 {bad}/{N}  (0 이어야 함)")
    assert bad == 0, "데이터 순서 또는 채점기가 다르다 — 진행하지 않는다"
else:
    f = OUT + f"{TAG}_base.parquet"
    if os.path.exists(f): B0 = pd.read_parquet(f)
    else:
        rows = []
        for k in range(N_FOLDS):
            rids = FOLD_ROWS[k]; rows += list(zip(rids, gen(PROMPTS[rids].tolist())))
        B0 = score([v[0] for _, v in rows], [r for r, _ in rows]); B0.to_parquet(f, index=False)
BASE = B0.em.values.astype(float)

# K 라벨 (closed-book)
if FR["closed_book"]:
    cb = pd.read_parquet(FR["closed_book"])
else:
    f = OUT + f"{TAG}_closed_book.parquet"
    if os.path.exists(f): cb = pd.read_parquet(f)
    else:
        qids = list(dict.fromkeys(m.qid)); qq = dict(zip(m.qid, m.question))
        cbp = [build(qq[x], None) for x in qids]
        order = np.argsort([len(p) for p in cbp], kind="stable"); res = [None] * len(cbp)
        for i, v in zip(order, gen([cbp[i] for i in order])): res[i] = v
        cb = pd.DataFrame(dict(qid=qids, raw=[r[0] for r in res]))
        cb["pred"] = [trunc(x) for x in cb.raw]
        g_of = dict(zip(m.qid, m.gold))
        cb["kplus"] = [int(any(PP.norm(p) == PP.norm(g) for g in g_of[x])) for p, x in zip(cb.pred, cb.qid)]
        cb.to_parquet(f, index=False)
yK = m.qid.map(dict(zip(cb.qid, cb.kplus))).values.astype(int)
CELL = np.array([("K+" if k else "K-") + c for k, c in zip(yK, m.C)])
print(f"base EM {BASE.mean():.4f} | C+ {BASE[yC].mean():.4f} | C- {BASE[isCm].mean():.4f}")
print("K+", f"{yK[yC].mean():.3f}", "| 셀", dict(zip(*np.unique(CELL, return_counts=True))))

base EM 0.4575 | C+ 0.7570 | C- 0.1580
K+ 0.384 | 셀 {np.str_('K+C+'): np.int64(384), np.str_('K+C-'): np.int64(384), np.str_('K-C+'): np.int64(616), np.str_('K-C-'): np.int64(616)}


## S6. packed 엔진 (p8s `ours_packed.py` 와 같은 연산)

In [ ]:
class Packed:
    """훅 3종(분기 마스크·A분기 steering·끝 위치 캡처)을 with 블록 동안만 붙인다."""

    def __init__(self, layers=LATE):
        self.L, self.dev = list(layers), model.device
        self.max_new = int(PP.MAX_NEW_TOKENS)
        self.eos = set(int(x) for x in EOS) | {int(tok.pad_token_id)}
        enc = lambda s: tok(s, add_special_tokens=False).input_ids
        self.IDS = [enc(p) for p in PROMPTS]
        cache = {}
        self.IDS_CB = []
        for qq in m.question:
            if qq not in cache: cache[qq] = enc(build(qq, None))
            self.IDS_CB.append(cache[qq])
        bos = tok.bos_token_id
        assert all(i[0] == bos and i[1] != bos for i in self.IDS + self.IDS_CB), "BOS 중복/누락"
        self.QI = m.qid.astype("category").cat.codes.values.astype(int)   # 질문 단위 묶음
        self.nQ = int(self.QI.max()) + 1
        self.ROWS_Q = {x: np.where(self.QI == x)[0].tolist() for x in range(self.nQ)}
        self.MASK, self.STEER = {"m": None}, {"on": False, "delta": None, "pos": None, "dec": None}
        self.CAP, self._h = {"on": False, "iA": None, "iB": None, "hA": {}, "hB": {}}, []

    def __enter__(self):
        Ls = model.model.layers
        self._h = [Ls[l].self_attn.register_forward_pre_hook(self._mask_hook, with_kwargs=True)
                   for l in range(len(Ls))]
        self._h += [Ls[l].register_forward_hook(self._steer_hook(j)) for j, l in enumerate(self.L)]
        self._h += [Ls[l].register_forward_hook(self._cap_hook(l)) for l in self.L]
        return self

    def __exit__(self, *e):
        for h in self._h: h.remove()
        self._h, self.MASK["m"] = [], None
        self.STEER["on"] = self.CAP["on"] = False
        return False

    def _mask_hook(self, mod, args, kwargs):
        if self.MASK["m"] is None: return None
        kwargs["attention_mask"] = self.MASK["m"]
        return args, kwargs

    def _steer_hook(self, j):
        def hook(mod, inp, out):
            if not self.STEER["on"]: return out
            h = out[0] if isinstance(out, (tuple, list)) else out
            d = self.STEER["delta"][:, j][:, None, :]
            mm = (self.STEER["pos"] if h.shape[1] > 1 else self.STEER["dec"])[:, :, None]
            h += (d * mm).to(h.dtype)
            return out
        return hook

    def _cap_hook(self, l):
        def hook(mod, inp, out):
            if not self.CAP["on"]: return out
            h = out[0] if isinstance(out, (tuple, list)) else out
            if h.shape[1] > 1:
                r = torch.arange(h.shape[0], device=h.device)
                self.CAP["hA"][l] = h[r, self.CAP["iA"]].float()
                self.CAP["hB"][l] = h[r, self.CAP["iB"]].float()
            return out
        return hook

    def pack(self, b):
        A_, B_ = [self.IDS[r] for r in b], [self.IDS_CB[r][1:] for r in b]
        T, n = max(len(a) + len(x) for a, x in zip(A_, B_)), len(b)
        ids = torch.full((n, T), tok.pad_token_id, dtype=torch.long)
        pos = torch.zeros((n, T), dtype=torch.long)
        mask = torch.zeros((n, 1, T, T), dtype=torch.bool)
        endA, endB = [], []
        for i, (a, x) in enumerate(zip(A_, B_)):
            nA, nB = len(a), len(x); pad = T - nA - nB
            ids[i, pad:pad + nA] = torch.tensor(a); ids[i, pad + nA:] = torch.tensor(x)
            pos[i, pad:pad + nA] = torch.arange(nA); pos[i, pad + nA:] = torch.arange(1, nB + 1)
            sa, sb = slice(pad, pad + nA), slice(pad + nA, T)
            mask[i, 0, sa, sa] = torch.ones(nA, nA, dtype=torch.bool).tril()
            mask[i, 0, sb, sb] = torch.ones(nB, nB, dtype=torch.bool).tril()
            mask[i, 0, sb, pad] = True
            for p in range(pad): mask[i, 0, p, p] = True
            endA.append(pad + nA - 1); endB.append(T - 1)
        return ids, pos, mask, endA, endB

    def stream(self, seed):
        order = np.random.RandomState(seed).permutation(np.arange(self.nQ))
        return [sum((self.ROWS_Q[x] for x in order[i:i + Q_PER_BATCH]), [])
                for i in range(0, self.nQ, Q_PER_BATCH)]

    @torch.inference_mode()
    def gen(self, b, rho=0.0, Wrow=None, Mrow=None):
        dev = self.dev
        ids, pos, mask, endA, endB = self.pack(b)
        n, T = ids.shape
        nA = [len(self.IDS[r]) for r in b]; nB = [len(self.IDS_CB[r]) - 1 for r in b]
        pads = [T - x - y for x, y in zip(nA, nB)]
        if rho > 0:
            self.STEER["delta"] = torch.tensor(rho * Mrow[:, :, None] * Wrow, dtype=torch.float32, device=dev)
            spos = torch.zeros((n, T), device=dev)
            for i in range(n): spos[i, pads[i] + 1:pads[i] + nA[i]] = 1.0       # A 구간, BOS 제외
            self.STEER.update(on=True, pos=spos, dec=torch.ones((n, 1), device=dev))
        self.MASK["m"] = mask.to(dev)
        self.CAP.update(on=True, iA=torch.tensor(endA, device=dev), iB=torch.tensor(endB, device=dev))
        try:
            o = model.model(input_ids=ids.to(dev), position_ids=pos.to(dev), use_cache=True)
            self.CAP["on"] = False
            hA = torch.stack([self.CAP["hA"][l] for l in self.L], 1)
            hB = torch.stack([self.CAP["hB"][l] for l in self.L], 1)
            if rho > 0: hA = hA - self.STEER["delta"]                         # pre-Δ 근사
            h, pkv = o.last_hidden_state, o.past_key_values
            r = torch.arange(n, device=dev)
            la = model.lm_head(h[r, endA]).float().log_softmax(-1)
            lb = model.lm_head(h[r, endB]).float().log_softmax(-1)
            ta, tb = la.argmax(-1), lb.argmax(-1)
            useB = ((ta != tb) & (la[r, tb] > lb[r, ta])).cpu().numpy()        # X1
            if rho > 0:
                self.STEER["dec"] = torch.tensor(~useB, dtype=torch.float32, device=dev)[:, None]
            allow = torch.zeros((n, T), dtype=torch.bool); nxt = torch.zeros(n, dtype=torch.long)
            for i in range(n):
                if useB[i]:
                    allow[i, pads[i]] = True; allow[i, pads[i] + nA[i]:] = True; nxt[i] = nB[i] + 1
                else:
                    allow[i, pads[i]:pads[i] + nA[i]] = True; nxt[i] = nA[i]
            allow, nxt = allow.to(dev), nxt.to(dev)
            tk = torch.where(torch.tensor(useB, device=dev), tb, ta)
            eos_t = torch.tensor(sorted(self.eos), device=dev)
            out, done = [tk], torch.isin(tk, eos_t)
            for t in range(1, self.max_new):
                if done.all(): break
                allow = torch.cat([allow, torch.ones((n, 1), dtype=torch.bool, device=dev)], 1)
                self.MASK["m"] = allow[:, None, None, :]
                o = model.model(input_ids=tk[:, None], position_ids=(nxt + t - 1)[:, None],
                                past_key_values=pkv, use_cache=True)
                pkv = o.past_key_values
                tk = model.lm_head(o.last_hidden_state[:, -1]).argmax(-1)
                tk = torch.where(done, torch.full_like(tk, tok.pad_token_id), tk)
                out.append(tk); done = done | torch.isin(tk, eos_t)
        finally:
            self.MASK["m"] = None; self.STEER["on"] = self.CAP["on"] = False
        g = torch.stack(out, 1).cpu().numpy()
        k = [next((j for j, x in enumerate(row) if int(x) in self.eos), len(row)) for row in g]
        raws = [tok.decode(row[:kk].tolist(), **DEC) for row, kk in zip(g, k)]
        return raws, useB, k, hA.cpu().numpy(), hB.cpu().numpy()

    def run(self, seed, online):
        S = np.zeros((len(self.L), model.config.hidden_size)); cnt, norms, rows = 0, [], []
        B = self.stream(seed)
        for bi, b in enumerate(B):
            if not online or cnt == 0:
                Wrow = Mrow = None; rho = 0.0
            else:
                w = -S / cnt; w /= np.linalg.norm(w, axis=-1, keepdims=True)
                Wrow = np.repeat(w[None], len(b), 0)
                Mrow = np.repeat(np.median(np.array(norms), 0)[None], len(b), 0); rho = RHO
            raws, useB, k, hA, hB = self.gen(b, rho, Wrow, Mrow)
            if online:                                   # 처리 후에만 누적 (자기 행 미사용)
                S += (hA - hB).sum(0); cnt += len(b); norms += list(np.linalg.norm(hA, axis=-1))
            rows += [dict(rid=int(r), raw=x, n_tok=int(kk), useB=bool(u), step=bi)
                     for r, x, u, kk in zip(b, raws, useB, k)]
            if (bi + 1) % 25 == 0: print(f"    {bi + 1}/{len(B)} 묶음")
        d = pd.DataFrame(rows).sort_values("rid").reset_index(drop=True)
        assert (d.rid.values == np.arange(N)).all()
        return d

PK = Packed()
print(f"packed 준비 | 질문 {PK.nQ} | 묶음 {len(PK.stream(0))}개 | 추가 prefill 토큰 "
      f"+{np.mean([len(x) - 1 for x in PK.IDS_CB]) / np.mean([len(x) for x in PK.IDS]):.1%}")

packed 준비 | 질문 1000 | 묶음 125개 | 추가 prefill 토큰 +36.7%


## S7. 실행 — packed_x1 + packed_online seed 0/1/2

A100: ConFiQA 1,000질문 런당 약 2분(총 8분) / NQ-Swap 은 문맥이 길어 1.5~2배.
L4: 각각 약 2.5배. 체크포인트(`hk/results_ext/{TAG}_packed_*.parquet`)가 있으면 건너뛴다.

In [ ]:
PKR = {}
def run_packed(name, seed, online):
    f = OUT + f"{TAG}_{name}.parquet"
    if os.path.exists(f) and len(pd.read_parquet(f)) == N:
        d = pd.read_parquet(f); print(f"[{name}] 체크포인트 재사용")
    else:
        t0 = time.time()
        with PK:
            d = PK.run(seed, online)
        s = score(d.raw.tolist(), d.rid.values)
        d = d.merge(s[["rid", "pred", "em", "acc_x", "rep3"]], on="rid")
        d.to_parquet(f, index=False); print(f"   ({(time.time() - t0) / 60:.1f}분)")
    PKR[name] = d
    e = d.em.values
    print(f"[{name}] EM {e.mean():.4f} | C+ {e[yC].mean():.4f} | C- {e[isCm].mean():.4f} "
          f"| B선택 {d.useB.mean():.3f} | hit {(d.n_tok >= PP.MAX_NEW_TOKENS).mean():.3f} rep3 {d.rep3.mean():.3f}")

run_packed("packed_x1", 0, online=False)
for s in SEEDS:
    run_packed(f"packed_online_s{s}", s, online=True)

    25/125 묶음
    50/125 묶음
    75/125 묶음
    100/125 묶음
    125/125 묶음
   (4.2분)
[packed_x1] EM 0.5030 | C+ 0.7000 | C- 0.3060 | B선택 0.275 | hit 0.003 rep3 0.002
    25/125 묶음
    50/125 묶음
    75/125 묶음
    100/125 묶음
    125/125 묶음
   (4.5분)
[packed_online_s0] EM 0.5185 | C+ 0.6700 | C- 0.3670 | B선택 0.277 | hit 0.004 rep3 0.003
    25/125 묶음
    50/125 묶음
    75/125 묶음
    100/125 묶음
    125/125 묶음
   (4.9분)
[packed_online_s1] EM 0.5200 | C+ 0.6720 | C- 0.3680 | B선택 0.274 | hit 0.004 rep3 0.004
    25/125 묶음
    50/125 묶음
    75/125 묶음
    100/125 묶음
    125/125 묶음
   (5.0분)
[packed_online_s2] EM 0.5205 | C+ 0.6740 | C- 0.3670 | B선택 0.276 | hit 0.004 rep3 0.003


## S8. 비교표 · 판정

판정 기준은 다영 노트북과 같다.
- **개선**: base 대비 전체 ΔEM 95% CI 하한 > 0
- **C+ 비열등**: base 대비 C+ ΔEM 95% CI 하한 > −2%p
- `vs ②` · `vs ①` 열은 같은 행끼리 paired bootstrap

In [ ]:
C = {"base": BASE}
if FR["router"]:
    C["① router"] = pd.read_parquet(FR["router"]).sort_values("rid").em.values.astype(float)
if FR["always"]:
    C["② always ρ0.06"] = pd.read_parquet(FR["always"]).sort_values("rid").em.values.astype(float)
C["packed_x1"] = PKR["packed_x1"].em.values.astype(float)
for s in SEEDS:
    C[f"packed_online_s{s}"] = PKR[f"packed_online_s{s}"].em.values.astype(float)
C["③ packed (seed avg)"] = np.mean([C[f"packed_online_s{s}"] for s in SEEDS], 0)
for k, v in C.items(): assert len(v) == N, k

refs = [r for r in ["② always ρ0.06", "① router"] if r in C]
cells_ = ["K+C+", "K+C-", "K-C+", "K-C-"]
f2 = lambda v, lo, hi: f"{v:+.2f} [{lo:+.2f}, {hi:+.2f}]"
rows = []
for n, e in C.items():
    dd = e - BASE
    lo, hi = ci(dd) * 100; plo = ci(dd, yC)[0] * 100
    r = dict(method=n, EM=e.mean() * 100, dC_plus=dd[yC].mean() * 100, dC_plus_lo=plo,
             dC_minus=dd[isCm].mean() * 100, vs_base=f2(dd.mean() * 100, lo, hi),
             improved=lo > 0, safe=plo > -2.0)
    for rf in refs:
        dr = e - C[rf]; a, b_ = ci(dr) * 100
        r["vs " + rf.split()[0]] = "기준" if n == rf else f2(dr.mean() * 100, a, b_)
    for c in cells_: r[c] = dd[CELL == c].mean() * 100
    rows.append(r)
T = pd.DataFrame(rows)

mk = lambda x: "O" if x else "X"
hdr = ["방법", "EM", "C+ Δ [하한]", "C− Δ", "vs base [95% CI]"] + ["vs " + rf.split()[0] for rf in refs] + ["개선", "C+ 비열등"]
lines = ["| " + " | ".join(hdr) + " |", "|" + "---|" * len(hdr)]
for _, r in T.sort_values("EM", ascending=False).iterrows():
    b0 = r.method == "base"
    cells_txt = [r.method, f"{r.EM:.2f}", f"{r.dC_plus:+.2f} [{r.dC_plus_lo:+.2f}]", f"{r.dC_minus:+.2f}",
                 "기준" if b0 else r.vs_base] + ["—" if b0 else r["vs " + rf.split()[0]] for rf in refs] + \
                ["—" if b0 else mk(r.improved), mk(r.safe)]
    lines.append("| " + " | ".join(cells_txt) + " |")
out_md = "\n".join(lines)
print(out_md)
print("\n[셀별 ΔEM vs base]  n =", {c: int((CELL == c).sum()) for c in cells_})
print(T[["method"] + cells_].round(2).to_string(index=False))

pk = C["③ packed (seed avg)"]
print("\n" + "=" * 64)
for rf in refs:
    dr = pk - C[rf]; a, b_ = ci(dr) * 100
    print(f"③ − {rf:14s} 전체 {f2(dr.mean()*100, a, b_)} | C− {dr[isCm].mean()*100:+.2f} | C+ {dr[yC].mean()*100:+.2f}")
dx = pk - C["packed_x1"]; a, b_ = ci(dx) * 100
print(f"steering 추가분 (③ − x1)     {f2(dx.mean()*100, a, b_)}")

T.to_csv(OUT + f"{TAG}_packed_table.csv", index=False)
open(OUT + f"{TAG}_packed_table.md", "w").write(out_md + f"""

데이터셋 {DATASET} ({m.qid.nunique()}질문 / {N}행, LIMIT={LIMIT}). base·K·②·① 은 다영 체크포인트(행 단위 paired).
③ = packed + A분기 온라인 w_ctx− (ρ0.04, L14–18) + X1, p8s 동결 사양 그대로(이 데이터에서 튜닝 없음). seed 0/1/2 평균.
질문 단위 bootstrap 2,000회. 개선 = vs base ΔEM 95% CI 하한 > 0. C+ 비열등 = C+ ΔEM 하한 > -2%p.
gold = 사실. B 분기 프롬프트 = PP.build_messages(q, None) (다영 closed-book 과 같음).
""")
print("저장:", OUT + f"{TAG}_packed_table.md")

| 방법 | EM | C+ Δ [하한] | C− Δ | vs base [95% CI] | 개선 | C+ 비열등 |
|---|---|---|---|---|---|---|
| packed_online_s2 | 52.05 | -8.30 [-10.38] | +20.90 | +6.30 [+4.37, +8.12] | O | X |
| packed_online_s1 | 52.00 | -8.50 [-10.57] | +21.00 | +6.25 [+4.32, +8.15] | O | X |
| ③ packed (seed avg) | 51.97 | -8.50 [-10.52] | +20.93 | +6.22 [+4.31, +8.04] | O | X |
| packed_online_s0 | 51.85 | -8.70 [-10.83] | +20.90 | +6.10 [+4.20, +7.92] | O | X |
| packed_x1 | 50.30 | -5.70 [-7.39] | +14.80 | +4.55 [+2.97, +6.23] | O | X |
| base | 45.75 | +0.00 [+0.00] | +0.00 | 기준 | — | O |

[셀별 ΔEM vs base]  n = {'K+C+': 384, 'K+C-': 384, 'K-C+': 616, 'K-C-': 616}
             method  K+C+  K+C-   K-C+  K-C-
               base  0.00  0.00   0.00  0.00
          packed_x1  1.56 41.67 -10.23 -1.95
   packed_online_s0  2.60 51.56 -15.75  1.79
   packed_online_s1  2.34 51.04 -15.26  2.27
   packed_online_s2  2.60 51.04 -15.10  2.11
③ packed (seed avg)  2.52 51.22 -15.37  2.06

steering 추가분 (③ − x1)     +1.67 [+0

## 읽는 법

| 관찰 | 해석 |
|---|---|
| ③ 개선 O · C+ 비열등 O | p8s 사양이 그대로 전이된다. Main 표에 싣는다 |
| ③ − ② 하한 > 0 | always steering 보다 유의하게 높다 |
| ③ − ① 하한 > 0 | routing 보다 유의하게 높다 (①은 이 데이터에서 C+ 를 지킨 유일한 기준선) |
| ③ C+ 실패 | 셀 분해에서 K−C+ 를 먼저 본다. ②의 손실이 몰린 칸이다 |

- p8s 에서 ③ − x1(steering 추가분)은 +2.37 [+1.33, +3.46] 이었다
- 결과가 나쁘게 나와도 ρ 를 바꿔 다시 돌리지 않는다. 바꾸면 외부 검증이 아니라 튜닝이 된다
- NQ-Swap 은 치환 유형 `corpus` 만 쓴다(다영과 동일). 문맥이 길어 OOM 이면 `Q_PER_BATCH = 4` 로 바꾸고 보고서에 적는다


In [ ]:
## Cell N1 — n300 짝 비교: packed vs 다영 ② transfer·CAD (S8 다음, CPU 수 초)
DY = FRIEND + "sota_confiqa_nqswap_always_steer/"
n3 = 600                                   # Cell N1: 앞 300질문 = rid 0–599 (C+, C− 교차)
SUB = {"② always transfer ρ0.06 (다영)": DY + "n300_result/nqswap_n300_retain_transfer_r0.06.parquet",
       "CAD (다영)":                      DY + "n300_result/nqswap_n300_cad.parquet",
       "base (다영 n300)":                DY + "nqswap_n300_base.parquet"}

# Cell N1: 다영 raw를 이 노트북 채점기로 재채점 → 채점 기준 통일
E = {}
for k, p in SUB.items():
    d = pd.read_parquet(p).sort_values("rid").reset_index(drop=True)
    assert (d.rid.values == np.arange(n3)).all(), k
    s = score(d.raw.tolist(), d.rid.values)
    print(f"{k}: 재채점 em 불일치 {int((s.em.values != d.em.values).sum())}/{n3}")
    E[k] = s.em.values.astype(float)
E["base (우리, 앞 300질문)"] = BASE[:n3]
E["packed_x1"] = C["packed_x1"][:n3]
ref = C["③ packed (seed avg)"][:n3]
print("base 다영 vs 우리 em 불일치:", int((E["base (다영 n300)"] != E["base (우리, 앞 300질문)"]).sum()), "행")

# Cell N1: 300질문 단위 bootstrap (질문 문자열 기준, RandomState(0), 2,000회)
yC3 = yC[:n3]; q3 = m.question.values[:n3]
uq3 = np.unique(q3); qi3 = {u: np.where(q3 == u)[0] for u in uq3}
rs3 = np.random.RandomState(0)
BOOT3 = [np.concatenate([qi3[u] for u in rs3.choice(uq3, len(uq3))]) for _ in range(2000)]
def ci3(v, msk):
    return np.percentile([v[i][msk[i]].mean() for i in BOOT3], [2.5, 97.5]) * 100

ALL = np.ones(n3, bool); fmt = lambda v, msk: "{:+.2f} [{:+.2f}, {:+.2f}]".format(v[msk].mean()*100, *ci3(v, msk))
rows = [dict(조건="③ packed (seed avg)", EM=ref.mean(), Cp=ref[yC3].mean(), Cm=ref[~yC3].mean())]
for k, v in E.items():
    d = ref - v
    rows.append(dict(조건=k, EM=v.mean(), Cp=v[yC3].mean(), Cm=v[~yC3].mean(),
                     **{"③−이것 전체": fmt(d, ALL), "C+": fmt(d, yC3), "C−": fmt(d, ~yC3)}))
print(pd.DataFrame(rows).round(4).to_string(index=False))

② always transfer ρ0.06 (다영): 재채점 em 불일치 0/600
CAD (다영): 재채점 em 불일치 0/600
base (다영 n300): 재채점 em 불일치 0/600
base 다영 vs 우리 em 불일치: 6 행
                          조건     EM     Cp     Cm                 ③−이것 전체                     C+                      C−
         ③ packed (seed avg) 0.5100 0.6878 0.3322                     NaN                    NaN                     NaN
② always transfer ρ0.06 (다영) 0.5017 0.7400 0.2633    +0.83 [-1.91, +3.54]   -5.22 [-8.52, -2.01]   +6.89 [+3.03, +11.04]
                    CAD (다영) 0.3017 0.5767 0.0267 +20.83 [+15.94, +25.31] +11.11 [+5.26, +16.89] +30.56 [+25.00, +36.10]
              base (다영 n300) 0.4783 0.7900 0.1667    +3.17 [-0.32, +6.57] -10.22 [-14.30, -6.40] +16.56 [+11.70, +21.54]
          base (우리, 앞 300질문) 0.4817 0.7967 0.1667    +2.83 [-0.62, +6.25] -10.89 [-14.73, -7.33] +16.56 [+11.67, +21.62]
                   packed_x1 0.5000 0.7200 0.2800    +1.00 [-0.95, +2.95]   -3.22 [-5.82, -0.78]    +5.22 [+2.23, +8.36]


## 확인

In [ ]:
## Cell N2 — K−C+ 손실 해부: 셀별 B선택 + 잃은 행 표본 (CPU 수 초)
x1 = PKR["packed_x1"].set_index("rid").sort_index()
uB, ex, eb = x1.useB.values.astype(bool), x1.em.values, BASE
B0r = B0.set_index("rid").sort_index()

# Cell N2: 셀별 — B선택률, 선택 분기별 정답률, 잃은/얻은 행
print("셀 | n | B선택률 | base EM | x1 EM | x1 EM(B선택행) | x1 EM(A선택행) | 잃음 (그중 B선택) | 얻음")
for c in ["K+C+", "K-C+", "K+C-", "K-C-"]:
    k = CELL == c
    lost = (eb == 1) & (ex == 0) & k
    print(f"{c} | {k.sum()} | {uB[k].mean():.3f} | {eb[k].mean():.3f} | {ex[k].mean():.3f} | "
          f"{ex[k & uB].mean():.3f} | {ex[k & ~uB].mean():.3f} | {lost.sum()} ({(lost & uB).sum()}) | "
          f"{((eb == 0) & (ex == 1) & k).sum()}")

# Cell N2: K−C+에서 base 정답 → x1 오답 & B선택 행 20개 무작위 (seed 0)
idx  = np.where((CELL == "K-C+") & (eb == 1) & (ex == 0) & uB)[0]
pick = np.sort(np.random.RandomState(0).choice(idx, min(20, len(idx)), replace=False))
print(f"\n[K−C+ 잃은 행 중 B선택 {len(idx)}개 → 표본 {len(pick)}]")
for r in pick:
    print(f"\n[rid {r}] Q: {m.question[r]}\n  gold : {list(m.gold[r])[:5]}"
          f"\n  base : {B0r.raw[r]!r}\n  x1(B): {x1.raw[r]!r}")

셀 | n | B선택률 | base EM | x1 EM | x1 EM(B선택행) | x1 EM(A선택행) | 잃음 (그중 B선택) | 얻음
K+C+ | 384 | 0.036 | 0.891 | 0.906 | 1.000 | 0.903 | 2 (0) | 8
K-C+ | 616 | 0.187 | 0.674 | 0.571 | 0.009 | 0.701 | 65 (64) | 2
K+C- | 384 | 0.427 | 0.310 | 0.727 | 0.994 | 0.527 | 0 (0) | 160
K-C- | 616 | 0.416 | 0.063 | 0.044 | 0.008 | 0.069 | 15 (12) | 3

[K−C+ 잃은 행 중 B선택 64개 → 표본 20]

[rid 38] Q: region of antarctica where march of the penguins filmed?
  gold : ["Dumont d'Urville Station"]
  base : "Dumont d'Urville Station"
  x1(B): 'Antarctic Peninsula'

[rid 62] Q: who wrote it 's a long long way to pasadena?
  gold : ['John Young']
  base : 'John Young'
  x1(B): 'Jimmie Davis'

[rid 246] Q: how many seasons of ray donovan has there been?
  gold : ['5']
  base : '5'
  x1(B): '6 seasons'

[rid 252] Q: who was the first to say i 'm going to disney world?
  gold : ['Dick Rutan', 'Jeana Yeager']
  base : 'Jeana Yeager'
  x1(B): 'Walt Disney'

[rid 564] Q: how many players in football hall of fame?
  gold :